# AWS SageMaker Feature Store 2. Features for Live Predictions

Switching on the online store, looking up one customer's newest features in milliseconds, and using them to get a real-time prediction from a SageMaker endpoint, laid out exactly as the model saw them in training.

---

Setup values and reference snippets are in code cells for easy copying; they are not meant to be executed. Run the Python steps individually. Saved example outputs appear in Markdown below their code.


## **Agenda**

1. [Why an online store](#scrollTo=c003)
2. [Setup](#scrollTo=c004)
3. [Start the endpoint](#scrollTo=c007)
4. [A feature group with an online store](#scrollTo=c010)
5. [Reading one customer](#scrollTo=c017)
6. [A prediction at checkout](#scrollTo=c020)
7. [A customer the store has never seen](#scrollTo=c027)
8. [Tear down](#scrollTo=c030)
9. [Class summary](#scrollTo=c033)
10. [References and further reading](#scrollTo=c034)


## **Preface**

Last class built the training side of a feature store. Customer features went into a feature group, the offline store kept every row with its time, and a point-in-time join gave each past order only what was known when it was placed. A training job learned from that and scored an AUC of about 0.72.

This class builds the live side. An order arrives at checkout. The app knows the customer's id and has a few milliseconds. It needs that customer's features as they are right now, laid out exactly the way the model saw them in training, and an answer from the model. You will switch on the **online store**, read one customer from it, put last class's model behind a **real-time endpoint**, and join the two into a prediction for one order. Then you will see what the online store says about a customer it has never seen.

**[Please Note:]** Same workflow as last class. Section 2 gets a domain and a JupyterLab space and uploads this notebook into JupyterLab. Read each explanation, run the Python step with **Shift+Enter**, and talk through the example output beneath it. The displayed outputs come from the original recorded run, so IDs and timings in yours will differ. The endpoint bills by the hour from section 3 on, so run section 8 before you leave.

**By the end you will be able to do all of this.**

1. Say what the online store keeps, how it differs from the offline store, and when each is used.
2. Create a feature group with both stores and write rows into it.
3. Read one customer's features from the online store and time it.
4. Turn an order and its customer's features into the same 18 numbers the model was trained on, and get a prediction from an endpoint.
5. Handle a customer the store has never seen, the same way training did.

**Prerequisites.** Last class, Feature Store 1, especially feature groups, `put_record` and the 18-column layout. Real-time endpoints from the SageMaker classes.

## **1. Why an online store**

Last class read the offline store with Athena. One query took a few seconds, and it returned rows from a full history of files in S3. That is right for training, which needs every row from every date. It is wrong for checkout. The app needs one customer, now, in milliseconds, and only their newest row.

The **online store** is built for that. It is part of the same feature group, switched on with one setting. Every row you write goes to both stores. The offline store adds it to the history. The online store keeps only the **newest row for each customer**, judged by `event_time`, and answers a lookup by `customer_id` in milliseconds.

| | Offline store | Online store |
|---|---|---|
| What it keeps | Every row ever written | The newest row for each customer |
| How you read it | SQL through Athena, in seconds | `get_record` by id, in milliseconds |
| Used for | Building training sets | Live predictions |

![c14_live_path.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/451/original/c14_live_path.png?1793273313)

The picture is the plan for the class. Rows go into the feature group. At checkout, the app looks up the customer in the online store, lays the order and the features out as 18 numbers, and sends them to the endpoint. The one rule that matters is that those 18 numbers are built exactly the way training built them. Section 6 shows how.

## **2. Setup**

**[Please Note:]** Browser. Same steps as last class, sections 3.1 to 3.3. Make sure the account has a default VPC. The console search term is below.


In [ ]:
VPC

Use the region configured in the Python setup cell below.


In [ ]:
us-west-2

Open the following service from the console search bar.


In [ ]:
SageMaker AI

If **SageMaker AI → Domains** shows no domain, create one with **Quick setup**. Click **Open Studio** on the domain's **User profiles** tab, open **JupyterLab**, and run a space. If there is none, create one with the following name.


In [ ]:
features

Click **Open JupyterLab**, upload this notebook with the **Upload Files** button, and open it with the **Python 3 (ipykernel)** kernel.

![c14_notebook_open.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/460/original/c14_notebook_open.png?1793274013)

The first Python cell sets the names the class uses, finds the bucket and role quick setup made, and makes the three AWS clients the class needs.

**[Please Note:]** JupyterLab. Run the following Python cell.


In [ ]:
"""The names this notebook uses, the bucket and role quick setup made, and three AWS clients."""
import time

import boto3
import pandas as pd
from sagemaker.core.helper.session_helper import get_execution_role

REGION = "us-west-2"
FEATURE_GROUP = "returns-customers"
ENDPOINT = "returns-fs-rt"
ACCOUNT = boto3.client("sts").get_caller_identity()["Account"]
BUCKET = f"sagemaker-{REGION}-{ACCOUNT}"   # the bucket quick setup made
ROLE = get_execution_role()                # the role this notebook runs as

sm = boto3.client("sagemaker")                          # creates and describes SageMaker things
store = boto3.client("sagemaker-featurestore-runtime")  # reads and writes feature rows
s3 = boto3.client("s3")

print("bucket:", BUCKET)
print("role:  ", ROLE.split("/")[-1])

**Example output — saved from the original run.**

<pre>sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
bucket: sagemaker-us-west-2-111122223333
role:   AmazonSageMaker-ExecutionRole-20261029T094482
</pre>


Both names come back, so the notebook can reach AWS as quick setup's role. The `INFO` lines come from the SageMaker library, as last class.

## **3. Start the endpoint**

An endpoint takes several minutes to start, so it goes first and gets ready while the rest of the class runs.

It serves last class's honest model, the one trained on point-in-time rows. Last class's tear down deleted the model file from the bucket, so a copy of it is kept in a public GitHub repository, [pulkit-scaler/mlops-feature-store-2](https://github.com/pulkit-scaler/mlops-feature-store-2). It is the same kind of file a training job leaves behind, a `model.tar.gz` with one XGBoost model inside, trained on the 18 columns from last class.


In [ ]:
https://github.com/pulkit-scaler/mlops-feature-store-2

The cell copies the file into the bucket, then makes the three things the SageMaker classes made for every endpoint. A **model** points at the file and the XGBoost container. An **endpoint configuration** says what machine to run it on. The **endpoint** is the running machine itself.

**[Please Note:]** JupyterLab. Run this cell. It returns in a few seconds with the endpoint still `Creating`.


In [ ]:
"""Start a real-time endpoint for last class's model. It takes minutes, so it starts first."""
import urllib.request

from botocore.exceptions import ClientError

XGBOOST_IMAGE = "683313688378.dkr.ecr.us-west-2.amazonaws.com/sagemaker-xgboost:1.7-1"   # the container it was trained with
MODEL_URL = "https://raw.githubusercontent.com/pulkit-scaler/mlops-feature-store-2/main/model.tar.gz"
urllib.request.urlretrieve(MODEL_URL, "model.tar.gz")
s3.upload_file("model.tar.gz", BUCKET, "fs-model/model.tar.gz")

try:
    sm.describe_endpoint(EndpointName=ENDPOINT)
    print(ENDPOINT, "already exists")
except ClientError:
    sm.create_model(
        ModelName="returns-fs-model",
        PrimaryContainer={"Image": XGBOOST_IMAGE, "ModelDataUrl": f"s3://{BUCKET}/fs-model/model.tar.gz"},
        ExecutionRoleArn=ROLE,
    )
    sm.create_endpoint_config(
        EndpointConfigName="returns-fs-config",
        ProductionVariants=[{"VariantName": "AllTraffic", "ModelName": "returns-fs-model",
                             "InstanceType": "ml.m5.large", "InitialInstanceCount": 1}],
    )
    sm.create_endpoint(EndpointName=ENDPOINT, EndpointConfigName="returns-fs-config")

print(ENDPOINT, sm.describe_endpoint(EndpointName=ENDPOINT)["EndpointStatus"])

**Example output — saved from the original run.**

<pre>returns-fs-rt Creating
</pre>


The endpoint is `Creating`. Section 6 waits for it. Until then, the class builds the other half of the live path, the features.

## **4. A feature group with an online store**

### 4.1 The feature rows

The rows are the ones last class's `customer_features` made from the 12,000 orders, one row every time a customer's history changed. They are saved as a file on the course's file server, so this class does not have to compute them again.

**[Please Note:]** JupyterLab. Run this cell.

In [ ]:
"""Last class's feature rows: one row for every moment a customer's history changed."""
FEATURES_URL = "https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/397/original/customer_features.csv?1793266796"
rows = pd.read_csv(FEATURES_URL)

print(len(rows), "rows for", rows.customer_id.nunique(), "customers")
rows[rows.customer_id == "C-00001"].tail(3)

**Example output — saved from the original run.**

<pre>15598 rows for 1440 customers
</pre>

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>customer_id</th>
      <th>event_time</th>
      <th>orders_so_far</th>
      <th>returns_so_far</th>
      <th>return_rate</th>
      <th>tenure_days</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>22</th>
      <td>C-00001</td>
      <td>2026-08-31T00:00:00Z</td>
      <td>13</td>
      <td>10</td>
      <td>0.6111</td>
      <td>1112</td>
    </tr>
    <tr>
      <th>23</th>
      <td>C-00001</td>
      <td>2026-09-02T00:00:00Z</td>
      <td>14</td>
      <td>10</td>
      <td>0.5789</td>
      <td>1114</td>
    </tr>
    <tr>
      <th>24</th>
      <td>C-00001</td>
      <td>2026-09-14T00:00:00Z</td>
      <td>14</td>
      <td>11</td>
      <td>0.6316</td>
      <td>1126</td>
    </tr>
  </tbody>
</table>


The same 15,598 rows, for the same 1,440 customers. C-00001 has 25 of them, and the three newest are on screen. The very newest, from 14 September, says 14 orders, 11 returns, a return rate of 0.6316 and 1,126 days since signing up. Keep that row in mind for section 5.

### 4.2 Create the group with both stores

The group is the same as last class, with the same six columns, the same record identifier and the same event time. One setting is new, `OnlineStoreConfig`, which switches the online store on. The offline store stays on too. A real feature group usually feeds both training and live predictions, and every row written goes to both.

**[Please Note:]** JupyterLab. Run this cell. It takes about half a minute.

In [ ]:
"""Create the feature group again, this time with the online store switched on."""
COLUMNS = [("customer_id", "String"), ("event_time", "String"),
           ("orders_so_far", "Integral"), ("returns_so_far", "Integral"),
           ("return_rate", "Fractional"), ("tenure_days", "Integral")]

start = time.time()
try:
    sm.create_feature_group(
        FeatureGroupName=FEATURE_GROUP,
        RecordIdentifierFeatureName="customer_id",
        EventTimeFeatureName="event_time",
        FeatureDefinitions=[{"FeatureName": name, "FeatureType": kind} for name, kind in COLUMNS],
        OnlineStoreConfig={"EnableOnlineStore": True},   # the one new setting
        OfflineStoreConfig={"S3StorageConfig": {"S3Uri": f"s3://{BUCKET}/feature-store"}},
        RoleArn=ROLE,
    )
except sm.exceptions.ResourceInUse:
    print(FEATURE_GROUP, "already exists")

status = "Creating"
while status == "Creating":
    time.sleep(5)
    group = sm.describe_feature_group(FeatureGroupName=FEATURE_GROUP)
    status = group["FeatureGroupStatus"]

print("status:", status, "after", round(time.time() - start), "s")
print("online store:", group["OnlineStoreConfig"])
print("offline store:", group["OfflineStoreConfig"]["S3StorageConfig"]["ResolvedOutputS3Uri"])

**Example output — saved from the original run.**

<pre>status: Created after 21 s
online store: {&#x27;EnableOnlineStore&#x27;: True}
offline store: s3://sagemaker-us-west-2-111122223333/feature-store/111122223333/sagemaker/us-west-2/offline-store/returns-customers-1793273468/data
</pre>


The group says the online store is enabled, and the offline store folder is in the bucket as before.

Studio shows the difference too.

**[Please Note:]** Studio (the browser tab you opened JupyterLab from). In the left menu click **More**, then **Feature Store**. Find and open the group with the following name.


In [ ]:
returns-customers

Look at the **Details** tab. Come back to JupyterLab afterwards.

![c14_studio_feature_group.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/461/original/c14_studio_feature_group.png?1793274016)

Under **Store configuration**, the **Details** tab says **Online/Offline**. It also names the record identifier, `customer_id`, and the event time feature, `event_time`, which the class set in the cell above.

### 4.3 Write the rows

Writing is the same `put_record` as last class. Each row goes to both stores. Eight writes run at a time, so they arrive in a mixed-up order.

**[Please Note:]** JupyterLab. Run this cell. The first write is retried for up to two minutes, because a brand-new group can refuse writes for a few seconds after it says `Created`.


In [ ]:
"""Write every feature row, one call per row, eight calls at a time. Each row goes to both stores."""
from concurrent.futures import ThreadPoolExecutor


def put(row):
    record = [{"FeatureName": name, "ValueAsString": str(value)} for name, value in row.items()]
    store.put_record(FeatureGroupName=FEATURE_GROUP, Record=record)


first = rows.iloc[0].to_dict()
for attempt in range(24):
    try:
        put(first)
        break
    except store.exceptions.ResourceNotFound:
        time.sleep(5)

start = time.time()
with ThreadPoolExecutor(max_workers=8) as pool:
    list(pool.map(put, rows.to_dict("records")))
print("wrote", len(rows), "rows in", round(time.time() - start), "s")

**Example output — saved from the original run.**

<pre>wrote 15598 rows in 25 s
</pre>


All 15,598 rows went in, in 25 seconds. Each one went to both stores.

## **5. Reading one customer**

A lookup is one call, `get_record`, with the group's name and the customer's id. The cell asks for C-00001 five times and times each answer, then prints what came back.


In [ ]:
C-00001

**[Please Note:]** JupyterLab. Run this cell.


In [ ]:
"""Ask the online store for one customer's features, five times, and time each answer."""
for i in range(5):
    start = time.time()
    answer = store.get_record(FeatureGroupName=FEATURE_GROUP, RecordIdentifierValueAsString="C-00001")
    print(f"answer {i + 1}: {(time.time() - start) * 1000:.0f} ms")

print({feature["FeatureName"]: feature["ValueAsString"] for feature in answer["Record"]})

**Example output — saved from the original run.**

<pre>answer 1: 22 ms
answer 2: 8 ms
answer 3: 8 ms
answer 4: 9 ms
answer 5: 8 ms
{&#x27;customer_id&#x27;: &#x27;C-00001&#x27;, &#x27;event_time&#x27;: &#x27;2026-09-14T00:00:00Z&#x27;, &#x27;orders_so_far&#x27;: &#x27;14&#x27;, &#x27;returns_so_far&#x27;: &#x27;11&#x27;, &#x27;return_rate&#x27;: &#x27;0.6316&#x27;, &#x27;tenure_days&#x27;: &#x27;1126&#x27;}
</pre>


The first answer took 22 milliseconds, because the first call also opens the connection. The next four took 8 or 9 milliseconds each. That is the speed checkout needs. The Athena queries last class took seconds.

The answer is one row, C-00001's row from 14 September, the newest of their 25. All 25 were written, in a mixed-up order. The online store kept the one with the latest `event_time` and nothing else. That is the customer's history as of now, which is exactly what checkout needs.

Every value comes back as text, the way it went in. Turning them back into numbers is the live path's job, in the next section.

## **6. A prediction at checkout**

### 6.1 The endpoint

First check that the endpoint from section 3 is ready. The cell waits if it is not.

The endpoint name is below.


In [ ]:
returns-fs-rt

**[Please Note:]** JupyterLab. Run this cell. If the endpoint is still starting, it waits, checking every 30 seconds.


In [ ]:
"""Wait until the endpoint from section 3 is ready, then say how long it took to start."""
sm.get_waiter("endpoint_in_service").wait(EndpointName=ENDPOINT)
info = sm.describe_endpoint(EndpointName=ENDPOINT)
print(ENDPOINT, info["EndpointStatus"], "after",
      round((info["LastModifiedTime"] - info["CreationTime"]).total_seconds()), "s")

**Example output — saved from the original run.**

<pre>returns-fs-rt InService after 200 s
</pre>


The endpoint took 200 seconds to start, a little over three minutes, and it is ready now.

### 6.2 The live path

The live path has three steps, and each is a small function.

1. **`lookup`** asks the online store for the customer and returns their features as a dictionary. A customer it has never seen comes back as an empty dictionary. Section 7 is about that case.
2. **`model_row`** lays the order and the features out as 18 numbers. It must match `model_table` from last class exactly, in the same columns, the same order and the same fill for a customer with no history. The model only ever saw that layout. A row in any other order would still get an answer, just a wrong one, with no error.
3. **`predict`** joins the 18 numbers with commas, sends them to the endpoint as CSV and reads back the chance of a return.

**[Please Note:]** JupyterLab. Run this cell. It only defines the functions. Put it next to last class's `model_table` and compare the column lists.

In [ ]:
"""The live path: look the customer up, lay out the 18 numbers as training did, ask the endpoint."""
runtime = boto3.client("sagemaker-runtime")   # the client that calls endpoints

# The same columns, in the same order, as last class's model_table.
ORDER_COLS = ["price", "discount_pct", "delivery_days", "is_gift", "seller_rating"]
FEATS = ["orders_so_far", "returns_so_far", "return_rate", "tenure_days"]
CATEGORIES = ["apparel", "beauty", "books", "electronics", "home"]
PAYMENTS = ["card", "cod", "upi", "wallet"]
NO_HISTORY = {"orders_so_far": 0, "returns_so_far": 0, "return_rate": 0.2}   # how training filled a first order


def lookup(customer_id):
    answer = store.get_record(FeatureGroupName=FEATURE_GROUP, RecordIdentifierValueAsString=customer_id)
    return {feature["FeatureName"]: feature["ValueAsString"] for feature in answer.get("Record", [])}


def model_row(order, features):
    if not features:   # never seen: no orders, no returns, tenure from the signup date
        days = (pd.Timestamp(order["order_date"]) - pd.Timestamp(order["signup_date"])).days
        features = {**NO_HISTORY, "tenure_days": days}
    return ([order[c] for c in ORDER_COLS]
            + [float(features[c]) for c in FEATS]
            + [int(order["category"] == c) for c in CATEGORIES]
            + [int(order["payment_method"] == p) for p in PAYMENTS])


def predict(order):
    row = ",".join(str(value) for value in model_row(order, lookup(order["customer_id"])))
    answer = runtime.invoke_endpoint(EndpointName=ENDPOINT, ContentType="text/csv", Body=row)
    return float(answer["Body"].read()), row

### 6.3 One order, two customers

Here is an order at checkout, a book for 450 with a 5% discount, paid by card. The cell scores it twice, once as C-00001's order and once as C-00002's. The order is identical. Only the customer, and so the features from the online store, differ.

**[Please Note:]** JupyterLab. Run this cell.

In [ ]:
"""One order at checkout, scored for C-00001 and then, unchanged, for C-00002."""
order = {"order_date": "2026-10-01", "category": "books", "price": 450.0, "discount_pct": 5,
         "delivery_days": 3, "payment_method": "card", "is_gift": 0, "seller_rating": 4.4}

for customer in ["C-00001", "C-00002"]:
    start = time.time()
    chance, row = predict({**order, "customer_id": customer})
    print(f"{customer}   chance of return {chance:.3f}   ({(time.time() - start) * 1000:.0f} ms)")
    print("   row sent:", row)

**Example output — saved from the original run.**

<pre>C-00001   chance of return 0.653   (183 ms)
   row sent: 450.0,5,3,0,4.4,14.0,11.0,0.6316,1126.0,0,0,1,0,0,1,0,0,0
C-00002   chance of return 0.045   (45 ms)
   row sent: 450.0,5,3,0,4.4,5.0,1.0,0.2,1096.0,0,0,1,0,0,1,0,0,0
</pre>


The same book order comes out at 0.653 for C-00001 and 0.045 for C-00002. The first call took 183 milliseconds, because it opened the connections. The second took 45 milliseconds for the lookup and the prediction together.

Here is C-00001's row again, value by value, with where each one came from.

| Position | Value | Where it came from |
|---|---|---|
| 1 to 5 | `450.0, 5, 3, 0, 4.4` | The order, `price` to `seller_rating` |
| 6 to 9 | `14.0, 11.0, 0.6316, 1126.0` | The online store, C-00001's newest row |
| 10 to 14 | `0, 0, 1, 0, 0` | The order's category, `books` |
| 15 to 18 | `1, 0, 0, 0` | The order's payment method, `card` |

The complete CSV rows from the saved example are below. The table above explains the first customer's row.


**C-00001 — CSV payload**


In [ ]:
450.0,5,3,0,4.4,14.0,11.0,0.6316,1126.0,0,0,1,0,0,1,0,0,0

**C-00002 — CSV payload**


In [ ]:
450.0,5,3,0,4.4,5.0,1.0,0.2,1096.0,0,0,1,0,0,1,0,0,0

Only positions 6 to 9 differ between the two customers. C-00002 has 5 orders and 1 return, a rate of 0.2, which brings their chance down to about one in twenty. C-00001 returns most of what they buy, so the same book is more likely than not to come back. The shop could ask C-00001 to pay before shipping, and let C-00002 check out as usual.


## **7. A customer the store has never seen**

New customers place orders too. C-01500 signed up on 20 September and is placing their first order. The online store has no row for them, because nothing has happened to them yet. The cell first shows what `get_record` answers for them, then scores the same book order.


In [ ]:
C-01500

**[Please Note:]** JupyterLab. Run this cell.


In [ ]:
"""A customer the online store has never seen: what it answers, and what the live path sends."""
answer = store.get_record(FeatureGroupName=FEATURE_GROUP, RecordIdentifierValueAsString="C-01500")
print("keys in the answer:", list(answer))

chance, row = predict({**order, "customer_id": "C-01500", "signup_date": "2026-09-20"})
print(f"C-01500   chance of return {chance:.3f}")
print("   row sent:", row)

**Example output — saved from the original run.**

<pre>keys in the answer: [&#x27;ResponseMetadata&#x27;]
C-01500   chance of return 0.151
   row sent: 450.0,5,3,0,4.4,0.0,0.0,0.2,11.0,0,0,1,0,0,1,0,0,0
</pre>


The answer has no `Record` key at all, only the request's own metadata. It is not an error. Code that expects a record and reads `answer["Record"]` crashes at checkout, and code that quietly treats a missing value as 0 sends the model a row that training never had.

`lookup` turns the missing record into an empty dictionary, and `model_row` fills it the way last class's `model_table` filled an order with no feature row before it. That means no orders, no returns, a return rate of 0.2, and the tenure worked out from the signup date, 11 days. The row sent is the same row the model saw for every first order in training, and the answer is 0.151.


**C-01500 — CSV payload**


In [ ]:
450.0,5,3,0,4.4,0.0,0.0,0.2,11.0,0,0,1,0,0,1,0,0,0

### **Checkpoint 7.1**

Change the order in section 6.3 to an apparel item paid cash on delivery. Replace the category and payment-method entries in its order dictionary with these snippets.


In [ ]:
"category": "apparel"

In [ ]:
"payment_method": "cod"

Score it for C-00001, C-00002 and C-01500. Write down which moved the chance more, changing the order or changing the customer. **Why it matters.** It shows how much of the prediction comes from the online store, and so how much a stale or missing row there can cost.


## **8. Tear down**

The endpoint bills for every hour it runs, so it goes first. Then the endpoint configuration and the model, then the feature group, then the files in the bucket. The online store's rows go with the feature group.

**[Please Note:]** JupyterLab. Run this cell now, and once more about ten minutes later. The text after the cell says why.

In [ ]:
"""Delete the endpoint first, since it bills by the hour, then everything else the class made."""
for kind, delete, name in [
    ("endpoint", sm.delete_endpoint, {"EndpointName": ENDPOINT}),
    ("endpoint config", sm.delete_endpoint_config, {"EndpointConfigName": "returns-fs-config"}),
    ("model", sm.delete_model, {"ModelName": "returns-fs-model"}),
    ("feature group", sm.delete_feature_group, {"FeatureGroupName": FEATURE_GROUP}),
]:
    try:
        delete(**name)
        print("deleted", kind)
    except ClientError as e:
        print(kind, "not found:", e.response["Error"]["Code"])

for prefix in ["feature-store/", "fs-model/"]:
    keys = [item["Key"] for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=prefix)
            for item in page.get("Contents", [])]
    for i in range(0, len(keys), 1000):   # one delete call takes at most 1,000 files
        s3.delete_objects(Bucket=BUCKET, Delete={"Objects": [{"Key": k} for k in keys[i:i + 1000]]})
    print("removed", len(keys), "files under", prefix)

**Example output — saved from the original run.**

<pre>deleted endpoint
deleted endpoint config
deleted model
deleted feature group
removed 1 files under feature-store/
removed 1 files under fs-model/
</pre>


Everything the class made is deleted. Look at the `feature-store/` line, though. It removed only 1 file, an empty marker Feature Store writes when a group is created. The rows from section 4.3 had not reached the offline store yet. Last class they took about six minutes to arrive, and they keep arriving even after the group is deleted. That is why the cell runs a second time, about ten minutes from now, to remove them.

**[Please Note:]** Studio. Click **JupyterLab** at the top left. Find the space with the following name.


In [ ]:
features

On that row, click **Stop**, and wait until the status says **Stopped**. Then, in the SageMaker AI console, open **Deployments & inference → Endpoints** in the left menu and check that the list is empty.

![c14_endpoints_empty.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/462/original/c14_endpoints_empty.png?1793274019)


## **9. Class summary**

Training reads history. Checkout needs the present, for one customer, in milliseconds. The online store is the part of a feature group that serves that. It is switched on with one setting, every row written goes to it and to the offline store, and it keeps only the newest row for each customer by `event_time`, whatever order the writes arrive in. A `get_record` call answered in 8 or 9 milliseconds once the connection was open.

The live path joined it to the model. `lookup` read the customer's newest features. `model_row` laid them out with the order as the same 18 numbers, in the same order, that last class's `model_table` built for training. `predict` sent them to a real-time endpoint serving last class's honest model. The same book order came out at 0.653 for a customer who returns most of what they buy and 0.045 for one who rarely does, and all of the difference came from the online store.

A customer the store has never seen gets no record and no error. The live path filled them exactly the way training filled a first order, because a model can only be trusted on rows built the way it was trained.

## **10. References and further reading**

**AWS documentation**

- [Feature Store concepts](https://docs.aws.amazon.com/sagemaker/latest/dg/feature-store-concepts.html). Online and offline stores, record identifiers and event times.
- [Online store](https://docs.aws.amazon.com/sagemaker/latest/dg/feature-store-storage-configurations-online-store.html). What the online store keeps and how it is read.
- [GetRecord](https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_feature_store_GetRecord.html). The lookup call, and what it returns.
- [Real-time inference](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints.html). Endpoints, from the SageMaker classes.

**Practitioner reading**

- [Training-serving skew](https://developers.google.com/machine-learning/guides/rules-of-ml#training-serving_skew) in Google's Rules of Machine Learning. Why the live row must be built the way the training row was.
- [Feature Store, the definitive guide](https://www.hopsworks.ai/dictionary/feature-store) by Hopsworks. Online and offline stores from the side of a team that builds them.